# 00 — Import External LinkedIn Dataset

Use this notebook to pull a third-party LinkedIn job-postings dataset (e.g. the Kaggle "LinkedIn Job Postings 2024" archive) into the project, **filtered to the same 5 industries we are analyzing**.

## Workflow

1. **Upload your dataset** to `data/external/` in the JupyterLab file browser:
    - If the dataset is the Kaggle 4-file archive: drag the whole folder (`postings.csv`, `jobs/job_industries.csv`, `mappings/industries.csv`, `companies/...`) into `data/external/`.
    - If it's a single CSV: drop it into `data/external/` and the notebook will fall back to title-based filtering.
2. Set `EXTERNAL_DIR` below to the folder you uploaded.
3. Run all cells. Filtered per-industry CSVs land in `data/raw/postings_<industry>_<period>.csv` and a combined file at `data/raw/postings_all_<period>.csv`.

## What this notebook does

- Loads the dataset and any linked tables (`job_industries.csv`, `industries.csv`).
- Maps LinkedIn industry IDs to our 5 buckets:
    - `pharma_chem` = Biotechnology Research (12), Pharmaceutical Manufacturing (15), Chemical Manufacturing (54), Chemical Raw Materials (690), Biotechnology (3238)
    - `legal_services` = Law Practice (9), Legal Services (10)
    - `farming_forestry` = Farming (63), Farming/Ranching/Forestry (201), Forestry & Logging (298)
    - `insurance` = Insurance (42), Insurance Carriers (1725), Insurance Agencies (1737), Insurance Funds (1743)
    - `patent_ip` = no clean industry ID; filtered via title regex (`patent`, `intellectual property`, `IP`, `prior art`, `trademark`)
- Reshapes columns to match our `postings_<industry>.csv` schema so it drops straight into `02_preprocessing.ipynb`.
- Tags `source_platform = 'linkedin_<year>'` so the supplementary data is traceable.
- Auto-detects the time period from the data.

## 1. Configuration

In [ ]:
# --- Project-root path bootstrap (added by repo-reorg) ---
# Ensures cwd is the project root regardless of where this notebook was
# launched from (root, code/, JupyterLab tree, VS Code, etc.).
import os
from pathlib import Path
_here = Path.cwd().resolve()
while not (_here / 'data').exists() and _here.parent != _here:
    _here = _here.parent
os.chdir(_here)
PROJECT_ROOT = _here
# ---------------------------------------------------------

from pathlib import Path

# Where you uploaded the dataset. Can be a folder or a single CSV.
# Default: data/external/  (everything in there is considered)
EXTERNAL_DIR = Path('data/external')

# Where filtered, schema-matched CSVs land
OUT_DIR = Path('data/raw')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Override the period suffix if auto-detection picks the wrong year.
# Set to None to auto-detect from listed_time/date_posted.
PERIOD_SUFFIX_OVERRIDE = None   # e.g. '2024' or '2023H2'

# Whether to write per-industry CSVs (one each) AND a combined file.
WRITE_PER_INDUSTRY = True
WRITE_COMBINED     = True

print(f'External dir: {EXTERNAL_DIR.resolve()}')
print(f'Output dir:   {OUT_DIR.resolve()}')

if not EXTERNAL_DIR.exists():
    EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)
    print(f'\nCreated {EXTERNAL_DIR}. Upload your dataset there, then re-run from cell 2.')
elif not any(EXTERNAL_DIR.iterdir()):
    print(f'\n{EXTERNAL_DIR} is empty. Upload your dataset there before running the next cells.')
else:
    print('\nContents of EXTERNAL_DIR:')
    for p in sorted(EXTERNAL_DIR.rglob('*')):
        if p.is_file():
            print(f'  {p.relative_to(EXTERNAL_DIR)}  ({p.stat().st_size / 1024**2:.1f} MB)')

## 2. Industry Mapping (the 5 we analyze)

In [2]:
# LinkedIn industry-ID → our project bucket. Curated to exclude tangential
# categories (Law Enforcement, Courts of Law, Wholesale Chemical, Agricultural
# Construction Machinery) that share keywords but aren't part of our research.
INDUSTRY_ID_MAP = {
    'pharma_chem':      [12, 15, 54, 690, 3238],
    'legal_services':   [9, 10],
    'farming_forestry': [63, 201, 298],
    'insurance':        [42, 1725, 1737, 1743],
}

INDUSTRY_LABELS = {
    'pharma_chem':      'Pharmaceutical / Chemical Manufacturing',
    'legal_services':   'Legal Services',
    'farming_forestry': 'Farming, Ranching and Forestry',
    'insurance':        'Insurance',
    'patent_ip':        'Patent Analysis and IP Research',
}

# patent_ip has no clean LinkedIn industry; pick it up via title regex instead.
# Tested on the Kaggle dataset: catches Patent Attorney, IP Paralegal, Trademark
# Attorney, etc. while excluding generic "IP analyst" if context is unclear.
PATENT_TITLE_REGEX = r'\b(patent|intellectual property|\bIP\b|prior art|trademark|technology transfer)\b'

# Fallback when the dataset has no industry mapping at all — looser title
# patterns for each bucket. Used only if no job_industries.csv is found.
TITLE_FALLBACK_REGEX = {
    'pharma_chem':      r'\b(pharma(ceutical)?|biotech(nology)?|chemist|chemical|laboratory|clinical research|drug development|validation)\b',
    'legal_services':   r'\b(paralegal|legal (assistant|analyst|operations)|contract specialist|e-discovery|compliance|document review|attorney)\b',
    'farming_forestry': r'\b(agronomy|agriculture|farming|ranch|crop|soil|forestry|natural resources|wildlife|conservation)\b',
    'insurance':        r'\b(insurance|underwriter|claims|actuari|adjuster|risk (analyst|management)|policy analyst|broker)\b',
    'patent_ip':        PATENT_TITLE_REGEX,
}

import pandas as pd
print('5 industry buckets configured:')
for k, v in INDUSTRY_ID_MAP.items():
    print(f'  {k:20s}  industry_ids: {v}')
print(f'  {"patent_ip":20s}  (title regex)')

5 industry buckets configured:
  pharma_chem           industry_ids: [12, 15, 54, 690, 3238]
  legal_services        industry_ids: [9, 10]
  farming_forestry      industry_ids: [63, 201, 298]
  insurance             industry_ids: [42, 1725, 1737, 1743]
  patent_ip             (title regex)


## 3. Discover Files in `EXTERNAL_DIR`

In [3]:
def find_dataset_files(root: Path) -> dict:
    """
    Discover the three files we care about anywhere under root.
    Returns dict with keys: 'postings', 'job_industries', 'industries' (any may be None).
    """
    found = {'postings': None, 'job_industries': None, 'industries': None}
    if not root.exists():
        return found

    candidates = [p for p in root.rglob('*.csv') if p.is_file()]
    if not candidates:
        return found

    for p in candidates:
        name = p.name.lower()
        if name == 'job_industries.csv':
            found['job_industries'] = p
        elif name == 'industries.csv' and 'jobs' not in p.parts:
            found['industries'] = p
        elif name == 'postings.csv':
            found['postings'] = p

    # If no file named exactly 'postings.csv', pick the largest CSV with a description column
    if found['postings'] is None and candidates:
        candidates.sort(key=lambda p: p.stat().st_size, reverse=True)
        for p in candidates:
            if p == found['job_industries'] or p == found['industries']:
                continue
            try:
                head = pd.read_csv(p, nrows=1)
                if 'description' in head.columns:
                    found['postings'] = p
                    break
            except Exception:
                continue

    return found


files = find_dataset_files(EXTERNAL_DIR)
print('Discovered files:')
for k, v in files.items():
    if v is None:
        print(f'  {k:15s}  (not found)')
    else:
        print(f'  {k:15s}  {v.relative_to(EXTERNAL_DIR.parent)}  ({v.stat().st_size / 1024**2:.1f} MB)')

if files['postings'] is None:
    raise FileNotFoundError(f'No postings CSV found in {EXTERNAL_DIR}. Upload your data there and re-run.')

Discovered files:
  postings         external/archive (7)/postings.csv  (492.9 MB)
  job_industries   external/archive (7)/jobs/job_industries.csv  (2.4 MB)
  industries       external/archive (7)/mappings/industries.csv  (0.0 MB)


## 4. Load and Inspect

In [4]:
# The set of columns we try to pull. The loader keeps whatever is present.
CANDIDATE_COLS = [
    'job_id',
    'title',
    'company_name', 'company',
    'description',
    'location',
    'formatted_experience_level', 'seniority_level', 'experience_level',
    'formatted_work_type', 'work_type', 'employment_type',
    'listed_time', 'date_posted', 'original_listed_time', 'posted_date',
    'job_posting_url', 'application_url', 'url',
    'normalized_salary', 'salary',
]

# Discover which of the candidate cols actually exist
header = pd.read_csv(files['postings'], nrows=0).columns.tolist()
use_cols = [c for c in CANDIDATE_COLS if c in header]
print(f'Loading {len(use_cols)} of {len(header)} columns from postings file:')
print(f'  {use_cols}')

postings = pd.read_csv(files['postings'], usecols=use_cols, low_memory=False)
print(f'\nLoaded: {len(postings):,} rows')

# Normalize column names to our schema's preferred names
rename_map = {
    'company_name':                'company',
    'formatted_experience_level':  'seniority',
    'seniority_level':             'seniority',
    'experience_level':            'seniority',
    'formatted_work_type':         'employment_type',
    'work_type':                   'employment_type',
    'job_posting_url':             'source_url',
    'application_url':             'source_url',
    'url':                         'source_url',
}
for old, new in rename_map.items():
    if old in postings.columns and new not in postings.columns:
        postings.rename(columns={old: new}, inplace=True)

# Description completeness
if 'description' in postings.columns:
    desc = postings['description'].fillna('')
    print(f'\nDescription completeness:')
    print(f'  non-null:     {desc.str.len().gt(0).sum():,}')
    print(f'  > 100 chars:  {desc.str.len().gt(100).sum():,}')
    print(f'  mean length:  {int(desc.str.len().mean()):,} chars')

Loading 13 of 31 columns from postings file:
  ['job_id', 'title', 'company_name', 'description', 'location', 'formatted_experience_level', 'formatted_work_type', 'work_type', 'listed_time', 'original_listed_time', 'job_posting_url', 'application_url', 'normalized_salary']

Loaded: 123,849 rows

Description completeness:
  non-null:     123,842
  > 100 chars:  123,698
  mean length:  3,766 chars


In [5]:
# Date detection — try multiple possible time columns
DATE_CANDIDATES = ['listed_time', 'original_listed_time', 'date_posted', 'posted_date']

date_series = None
date_col_used = None
for c in DATE_CANDIDATES:
    if c not in postings.columns:
        continue
    s = postings[c]
    # If numeric and large, assume Unix epoch milliseconds
    if pd.api.types.is_numeric_dtype(s) and s.dropna().median() > 1e10:
        parsed = pd.to_datetime(s, unit='ms', errors='coerce')
    else:
        parsed = pd.to_datetime(s, errors='coerce')
    if parsed.notna().sum() > (date_series.notna().sum() if date_series is not None else 0):
        date_series = parsed
        date_col_used = c

if date_series is not None:
    postings['date_posted'] = date_series.dt.strftime('%Y-%m-%d')
    print(f'Date column used: {date_col_used}')
    print(f'  range:        {date_series.min()} → {date_series.max()}')
    print(f'  parsed:       {date_series.notna().sum():,} / {len(date_series):,}')
    detected_period = f'{date_series.min().year}'
    if date_series.max().year != date_series.min().year:
        detected_period = f'{date_series.min().year}-{date_series.max().year}'
else:
    postings['date_posted'] = ''
    print('No usable date column found — date_posted left blank.')
    detected_period = 'unknown'

PERIOD_SUFFIX = PERIOD_SUFFIX_OVERRIDE or detected_period
print(f'\nPeriod suffix to use in output filenames: "{PERIOD_SUFFIX}"')

Date column used: listed_time
  range:        2024-03-24 21:50:14 → 2024-04-20 00:26:56
  parsed:       123,849 / 123,849

Period suffix to use in output filenames: "2024"


## 5. Tag Each Row with Our Industry Bucket

In [6]:
import re

postings['industry_key'] = None

# --- Stage 1: industry-ID matching (if mapping table available) ---
tagged_by_id = 0
if files['job_industries'] is not None:
    job_ind = pd.read_csv(files['job_industries'])
    print(f'Loaded job_industries.csv  ({len(job_ind):,} rows, {job_ind["job_id"].nunique():,} unique jobs)')

    jobid_to_ours = {}
    for ours, ids in INDUSTRY_ID_MAP.items():
        matched = job_ind.loc[job_ind['industry_id'].isin(ids), 'job_id']
        for jid in matched:
            jobid_to_ours[jid] = ours
    postings['industry_key'] = postings['job_id'].map(jobid_to_ours)
    tagged_by_id = postings['industry_key'].notna().sum()
    print(f'Stage 1 (industry_id match): tagged {tagged_by_id:,} postings')
else:
    print('No job_industries.csv found — skipping Stage 1 (industry-ID matching).')

# --- Stage 2: title regex fallback ---
title_series = postings['title'].fillna('') if 'title' in postings.columns else pd.Series([''] * len(postings))

# Always apply the patent_ip title regex (overrides any prior tag)
patent_mask = title_series.str.contains(PATENT_TITLE_REGEX, regex=True, case=False, na=False)
postings.loc[patent_mask, 'industry_key'] = 'patent_ip'
patent_count = int(patent_mask.sum())
print(f'patent_ip title-regex match: {patent_count:,}')

# If Stage 1 was skipped entirely, also apply the broad title fallback for the other 4 buckets
if files['job_industries'] is None:
    print('\nUsing title-regex fallback for the other 4 industries (Stage 1 unavailable):')
    for ours, pattern in TITLE_FALLBACK_REGEX.items():
        if ours == 'patent_ip':
            continue
        mask = title_series.str.contains(pattern, regex=True, case=False, na=False)
        # don't overwrite patent_ip
        mask = mask & postings['industry_key'].isna()
        postings.loc[mask, 'industry_key'] = ours
        print(f'  {ours:20s}  {int(mask.sum()):,}')

# --- Drop rows not matching any of our 5 industries ---
tagged = postings.dropna(subset=['industry_key']).copy()
tagged['industry_label'] = tagged['industry_key'].map(INDUSTRY_LABELS)

print(f'\nFinal counts after tagging:')
print(tagged['industry_key'].value_counts().to_string())
print(f'\nTotal rows matching our 5 industries: {len(tagged):,}')

Loaded job_industries.csv  (164,808 rows, 127,125 unique jobs)
Stage 1 (industry_id match): tagged 8,587 postings
patent_ip title-regex match: 125

Final counts after tagging:
industry_key
pharma_chem         4221
insurance           2652
legal_services      1494
farming_forestry     135
patent_ip            125

Total rows matching our 5 industries: 8,627


/var/folders/x7/7kt002jd7ys7dblh20f3nbmw0000gn/T/ipykernel_58843/3872508124.py:26: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  patent_mask = title_series.str.contains(PATENT_TITLE_REGEX, regex=True, case=False, na=False)


## 6. Reshape to Project Schema and Save

In [7]:
from datetime import datetime
import hashlib

OUR_SCHEMA = [
    'job_id', 'title', 'company', 'industry_key', 'industry_label',
    'seniority', 'location', 'date_posted', 'description', 'skills_tags',
    'employment_type', 'source_platform', 'source_url', 'scraped_at',
]

# Fill missing schema columns with defaults
for col in OUR_SCHEMA:
    if col not in tagged.columns:
        tagged[col] = ''

# Coerce job_id to short hex string if it's numeric (matches our scraper's format)
if pd.api.types.is_numeric_dtype(tagged['job_id']):
    tagged['job_id'] = tagged['job_id'].astype('Int64').astype(str)
tagged['job_id'] = tagged['job_id'].fillna('').astype(str)

# Tag platform and scrape time
tagged['source_platform'] = f'linkedin_{PERIOD_SUFFIX}'
tagged['scraped_at']      = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
tagged['skills_tags']     = tagged.get('skills_tags', '').fillna('')

final = tagged[OUR_SCHEMA].copy()
print(f'Final dataframe: {len(final):,} rows × {len(final.columns)} cols (schema matched)')

# Write per-industry files
if WRITE_PER_INDUSTRY:
    print(f'\nPer-industry files → {OUT_DIR}:')
    for ind in INDUSTRY_LABELS:
        subset = final[final['industry_key'] == ind]
        if len(subset) == 0:
            continue
        path = OUT_DIR / f'postings_{ind}_{PERIOD_SUFFIX}.csv'
        subset.to_csv(path, index=False)
        print(f'  {path.name:50s}  {len(subset):>5,} rows  ({path.stat().st_size / 1024:.0f} KB)')

if WRITE_COMBINED:
    combined_path = OUT_DIR / f'postings_all_{PERIOD_SUFFIX}.csv'
    final.to_csv(combined_path, index=False)
    print(f'\nCombined → {combined_path.name}  ({len(final):,} rows, {combined_path.stat().st_size / 1024**2:.1f} MB)')

Final dataframe: 8,627 rows × 14 cols (schema matched)

Per-industry files → data/raw:
  postings_pharma_chem_2024.csv                       4,221 rows  (19516 KB)
  postings_legal_services_2024.csv                    1,494 rows  (4049 KB)
  postings_farming_forestry_2024.csv                    135 rows  (510 KB)
  postings_insurance_2024.csv                         2,652 rows  (11144 KB)
  postings_patent_ip_2024.csv                           125 rows  (377 KB)

Combined → postings_all_2024.csv  (8,627 rows, 34.8 MB)


## 7. Next Steps